# Лабораторная работа № 3. Вариант 24
## Классификация опухолей головного мозга по МРТ с помощью CNN

**Цель:** обучить свою свёрточную сеть, улучшить её настройки, сравнить с ResNet18 и MobileNetV3, затем разобрать верные и ошибочные предсказания.

**Датасет:** [Brain Tumor MRI Dataset (Kaggle)](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset). Четыре класса: **глиома**, **менингиома**, **без опухоли**, **опухоль гипофиза**.

**Этапы:** загрузка → проверка снимков → базовая CNN → улучшения CNN → перенос обучения → дообучение части свёрточных блоков → тест и анализ ошибок.

*Результаты учебные; они не предназначены для медицинской диагностики.*

### Запуск в Google Colab

1. Откройте ноутбук в [Google Colab](https://colab.research.google.com/) и включите **Среда выполнения → Сменить среду выполнения → T4 GPU**.
2. Нажмите **Среда выполнения → Выполнить все**, затем разрешите подключение Google Drive.
3. Датасет с Kaggle загрузится автоматически. Результаты сохранятся в **MyDrive/LR3_variant24_CNN_MRI**.

При повторном запуске **уже обученные модели не переобучаются**. Два новых эксперимента продолжают обучение ResNet18 и MobileNetV3 на основе прежних лучших весов. Если весов нет, ноутбук обучит исходные модели сначала. Обучение ограничено примерно **30 минутами**, но скорость загрузки данных и доступность GPU зависят от Colab.

### Шаг 1. Запуск среды
Устанавливаем библиотеку Kaggle, подключаем Python-библиотеки и проверяем видеокарту. Фиксируем случайное число `seed=42`, чтобы эксперименты было проще повторить.

In [ ]:
#@title 1. Библиотека для загрузки датасета
%pip -q install -U kagglehub

In [ ]:
#@title 2. Библиотеки и проверка GPU
import os, sys, json, time, random, hashlib, warnings, shutil, zipfile, platform
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
import kagglehub

# Одинаковый seed помогает воспроизвести случайные операции.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Для ускорения обучения используем видеокарту, если она доступна.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True  # фиксированный размер входа, ускорение свёрток
print('Python:', sys.version.split()[0], '| torch:', torch.__version__)
print('Устройство:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'не обнаружен')
if DEVICE.type != 'cuda':
    raise RuntimeError('Включите GPU: Среда выполнения → Сменить среду выполнения → T4 GPU. Не запускайте обучение на CPU.')

### Шаг 2. Основные параметры
Задаём размер изображения **160 × 160**, число эпох, папку Google Drive и ограничение обучения. Используем до **700 обучающих** и до **100 валидационных** снимков каждого класса; это позволяет быстрее провести несколько экспериментов. **Тест не сокращается ради экономии времени**: исключаются только точные дубликаты.

In [ ]:
#@title 3. Основные настройки и сохранение в Google Drive
# ==================== ОСНОВНЫЕ НАСТРОЙКИ ====================
USE_GOOGLE_DRIVE = True
OUT_FOLDER_NAME = 'LR3_variant24_CNN_MRI'
MANUAL_DATASET_PATH = ''  # необязательно: путь к каталогу с Training/ и Testing/
IMAGE_SIZE = 160           # 160 быстрее, чем 224; для всех моделей одинаковый размер
BATCH_SIZE = 32
NUM_WORKERS = 2
MAX_TRAIN_PER_CLASS = 700   # максимум 2800 изображений на обучении; 0 = без ограничения
MAX_VAL_PER_CLASS = 100     # максимум 400 на валидации; 0 = без ограничения
VAL_FRACTION = 0.12        # выделяется ТОЛЬКО из официального Training
MAX_TOTAL_MINUTES = 30     # предел этапа обучения; отдельные тесты занимают ещё несколько минут
EARLY_STOPPING_PATIENCE = 3
TRANSFER_EPOCHS = 8       # максимум для каждого дополнительного обучения ResNet/MobileNet
TRANSFER_PATIENCE = 3
FORCE_REBUILD_SPLIT = False  # True только если требуется намеренно создать новое разбиение
# Число эпох по каждому эксперименту (не меняйте во время продолжения незаконченных опытов)
EPOCHS = {'cnn_baseline': 8, 'cnn_bn': 6, 'cnn_bn_dropout': 7,
          'cnn_bn_dropout_aug': 10, 'resnet18': 6, 'mobilenet_v3_small': 6,
          'cnn_finetuned': 12}  # продолжение обучения baseline с меньшим LR
# ============================================================

SESSION_STARTED = time.monotonic()
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive') / OUT_FOLDER_NAME
else:
    OUT_DIR = Path('/content') / OUT_FOLDER_NAME
CKPT_DIR = OUT_DIR / 'checkpoints'
FIG_DIR = OUT_DIR / 'figures'
TABLE_DIR = OUT_DIR / 'tables'
for p in (OUT_DIR, CKPT_DIR, FIG_DIR, TABLE_DIR):
    p.mkdir(parents=True, exist_ok=True)
print('Каталог результатов:', OUT_DIR)
print('Папка для весов, графиков и таблиц готова.')

## 1. Загрузка и разделение изображений

Сначала находим четыре папки с видами снимков и проверяем файлы. В наборе уже есть `Training` и `Testing`: **тест оставляем отдельно**, а из `Training` выделяем обучение и валидацию.

- **Train:** изображения, по которым сеть учится.
- **Validation:** помогает выбрать настройки и лучшие веса.
- **Test:** нужен только для окончательной оценки всех моделей.

Проверка SHA-256 выявляет одинаковые файлы. Но она не обнаруживает все похожие снимки, а разделить данные по пациентам без их идентификаторов невозможно.

### Шаг 3. Получение датасета
Скачиваем снимки и проверяем, что в папках `Training` и `Testing` есть все четыре класса.

In [ ]:
#@title 4. Загрузка и проверка папок датасета
DATASET_ID = 'masoudnickparvar/brain-tumor-mri-dataset'
if MANUAL_DATASET_PATH.strip():
    candidate_root = Path(MANUAL_DATASET_PATH).expanduser()
    if not candidate_root.exists():
        raise FileNotFoundError(f'MANUAL_DATASET_PATH не существует: {candidate_root}')
else:
    try:
        candidate_root = Path(kagglehub.dataset_download(DATASET_ID))
    except Exception as exc:
        raise RuntimeError(
            'Не удалось автоматически скачать публичный датасет из Kaggle. '
            'Откройте https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset, '
            'скачайте и распакуйте архив в /content (либо Google Drive), затем задайте '
            'MANUAL_DATASET_PATH в ячейке 3. Оригинальная ошибка: ' + repr(exc)
        ) from exc

def locate_dataset_root(top):
    # Kagglehub может вернуть папку, содержащую дополнительные вложенные каталоги.
    roots = [top] + [p for p in top.rglob('*') if p.is_dir() and len(p.relative_to(top).parts) <= 3]
    matches = [p for p in roots if (p/'Training').is_dir() and (p/'Testing').is_dir()]
    if not matches:
        raise FileNotFoundError(f'Не обнаружены каталоги Training/ и Testing/ внутри {top}')
    return matches[0]

DATA_ROOT = locate_dataset_root(candidate_root)
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_RU = {'glioma':'Глиома', 'meningioma':'Менингиома',
            'notumor':'Без опухоли', 'pituitary':'Опухоль гипофиза'}
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
for subset in ('Training', 'Testing'):
    for label in CLASS_NAMES:
        if not (DATA_ROOT/subset/label).is_dir():
            raise FileNotFoundError(f'Отсутствует папка {subset}/{label} в {DATA_ROOT}')
print('Датасет найден:', DATA_ROOT)

### Шаг 4. Проверка и разделение
Проверяем, открываются ли снимки, убираем одинаковые файлы из обучающих данных и проверяем пересечения с тестом. Сохраняем разбиение в CSV: при повторном запуске используются **те же снимки**, что и для ранее обученных моделей. Точные повторы внутри теста уберём только на этапе окончательной оценки.

In [ ]:
#@title 5. Проверка файлов и разделение на train / val / test
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
MANIFEST_PATH = TABLE_DIR / 'split_manifest.csv'

def file_digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

# Проверяем, что изображения читаются, и вычисляем хеш каждого файла.
def scan_dataset():
    rows, damaged = [], []
    for part in ('Training', 'Testing'):
        for cls in CLASS_NAMES:
            paths = sorted(p for p in (DATA_ROOT/part/cls).rglob('*')
                           if p.is_file() and p.suffix.lower() in EXTENSIONS)
            for path in tqdm(paths, desc=f'{part}/{cls}', leave=False):
                try:
                    with Image.open(path) as im:
                        im.verify()
                    rows.append({'rel_path': path.relative_to(DATA_ROOT).as_posix(),
                                 'source': part, 'label': cls, 'sha256': file_digest(path)})
                except Exception as exc:
                    damaged.append((str(path), repr(exc)))
    if damaged:
        print(f'Предупреждение: повреждено файлов: {len(damaged)}. Примеры:', damaged[:3])
    frame = pd.DataFrame(rows)
    if frame.empty or any(frame.loc[frame.source == 'Training', 'label'].value_counts().get(c, 0) < 2
                          for c in CLASS_NAMES):
        raise RuntimeError('Недостаточно корректных данных для разделения на обучение и валидацию.')
    return frame

# При обычном перезапуске пользуемся теми же файлами; при обновлении датасета проверяем их наличие.
if MANIFEST_PATH.exists() and not FORCE_REBUILD_SPLIT:
    manifest = pd.read_csv(MANIFEST_PATH)
    if any(not (DATA_ROOT/rel).exists() for rel in manifest.rel_path):
        raise RuntimeError('Сохранённый manifest не соответствует текущей версии датасета. '
                           'Верните старую версию или установите FORCE_REBUILD_SPLIT=True, '
                           'понимая, что старые веса тогда нельзя честно сравнивать.')
    print('Использовано ранее зафиксированное разбиение:', MANIFEST_PATH)
else:
    all_files = scan_dataset()
    train_src = all_files.loc[all_files.source == 'Training'].copy()
    test_src = all_files.loc[all_files.source == 'Testing'].copy()
    repeated_test = int(test_src.duplicated('sha256').sum())
    if repeated_test:
        print(f'ВНИМАНИЕ: внутри официального Testing найдено {repeated_test} точных дубликатов; '
              'оставляем тест как в источнике, сообщаем ограничение в отчёте.')
    overlap = train_src.sha256.isin(set(test_src.sha256))
    print('Точные совпадения train/test, исключённые из Training:', int(overlap.sum()))
    train_src = train_src.loc[~overlap].drop_duplicates('sha256', keep='first').reset_index(drop=True)
    # Отдельный test не трогаем: валидацию выделяем только из Training.
    tr_idx, va_idx = train_test_split(
        np.arange(len(train_src)), test_size=VAL_FRACTION, stratify=train_src.label,
        random_state=SEED
    )
    def balanced_cap(frame, cap):
        if not cap or cap <= 0:
            return frame.sort_values('rel_path')
        return pd.concat([g.sample(n=min(cap, len(g)), random_state=SEED)
                          for _, g in frame.groupby('label')]).sort_values('rel_path')
    tr = balanced_cap(train_src.iloc[tr_idx].copy(), MAX_TRAIN_PER_CLASS)
    va = balanced_cap(train_src.iloc[va_idx].copy(), MAX_VAL_PER_CLASS)
    te = test_src.copy()
    tr['split'] = 'train'; va['split'] = 'val'; te['split'] = 'test'
    manifest = pd.concat([tr, va, te], ignore_index=True)
    manifest = manifest[['rel_path', 'source', 'label', 'sha256', 'split']]
    # Фиксируем список снимков, чтобы при повторном запуске сравнение было тем же.
    manifest.to_csv(MANIFEST_PATH, index=False)
    print('Сохранён новый manifest:', MANIFEST_PATH)

assert manifest.split.isin(['train', 'val', 'test']).all()
assert set(manifest.label) == set(CLASS_NAMES)
assert not manifest.loc[manifest.split == 'train', 'sha256'].isin(
    manifest.loc[manifest.split != 'train', 'sha256']).any(), 'Обнаружена утечка точных дубликатов из train!'
assert not manifest.loc[manifest.split == 'val', 'sha256'].isin(
    manifest.loc[manifest.split == 'test', 'sha256']).any(), 'Обнаружена утечка val/test!'
assert set(manifest.loc[manifest.split == 'test', 'source']) == {'Testing'}
assert set(manifest.loc[manifest.split != 'test', 'source']) == {'Training'}
SPLIT_ID = hashlib.sha256(manifest.sort_values('rel_path').to_csv(index=False).encode()).hexdigest()[:16]
print('ID разбиения:', SPLIT_ID)
print(pd.crosstab(manifest['label'], manifest['split']).reindex(CLASS_NAMES))
print('Дубликатов в тесте:', int(manifest.loc[manifest.split=='test'].duplicated('sha256').sum()))

### Шаг 5. Примеры снимков
Выводим несколько МРТ каждого класса. Это позволяет увидеть особенности исходных изображений до обработки.

In [ ]:
#@title 6. Примеры снимков каждого класса
examples = manifest[manifest.split=='train'].groupby('label', group_keys=False).sample(n=2, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for ax, (_, row) in zip(axes.flat, examples.sort_values('label').iterrows()):
    with Image.open(DATA_ROOT/row.rel_path) as im:
        ax.imshow(im.convert('RGB'))
    ax.set_title(CLASS_RU[row.label], fontsize=11)
    ax.axis('off')
plt.suptitle('Примеры изображений для обучения (до преобразований)', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR/'01_train_samples.png', dpi=160, bbox_inches='tight')
plt.show()

### Шаг 6. Изображения для сети
Изменяем размеры, переводим снимки в RGB и нормализуем пиксели. Для обучающих изображений при необходимости включаем слабую аугментацию; тестовые снимки оставляем без случайных изменений.

In [ ]:
#@title 7. Преобразование изображений и загрузчики
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
# Общая нормализация подходит и собственной CNN, и сетям ImageNet.
BASE_TRANSFORM = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
# Небольшие случайные преобразования применяются только при обучении.
TRAIN_AUG_TRANSFORM = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(degrees=8, fill=0),
    transforms.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.94, 1.06), fill=0),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class MRIDataset(Dataset):
    def __init__(self, frame, transform):
        self.entries = [(DATA_ROOT/rel, CLASS_TO_IDX[lab]) for rel, lab in
                        zip(frame.rel_path.tolist(), frame.label.tolist())]
        self.transform = transform
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, idx):
        path, label = self.entries[idx]
        with Image.open(path) as im:
            image = im.convert('RGB')  # предобученные RGB-модели; для всех одинаковый вход
        return self.transform(image), label

train_frame = manifest.loc[manifest.split=='train'].reset_index(drop=True)
val_frame = manifest.loc[manifest.split=='val'].reset_index(drop=True)
test_frame_original = manifest.loc[manifest.split=='test'].reset_index(drop=True)
# Повторы внутри официального теста исключаем лишь при оценке; обучение и SPLIT_ID не меняются.
test_frame = test_frame_original.drop_duplicates(subset=['sha256'], keep='first').reset_index(drop=True)
removed_test_duplicates = len(test_frame_original) - len(test_frame)

# DataLoader собирает изображения в небольшие пакеты; последние неполные пакеты сохраняем.
def loader(frame, augmentation=False, shuffle=False):
    ds = MRIDataset(frame, TRAIN_AUG_TRANSFORM if augmentation else BASE_TRANSFORM)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type=='cuda',
                      drop_last=False, persistent_workers=False)

VAL_LOADER = loader(val_frame)
TEST_LOADER = loader(test_frame)
print(f'train={len(train_frame)} | val={len(val_frame)} | test={len(test_frame)}')
print(f'Из официального теста исключено точных повторов: {removed_test_duplicates}')
print('Принципиально важно: val/test без аугментации, drop_last=False.')

## 2. Подготовка входа и архитектуры CNN

Все сети получают изображения **160×160 пикселей** с одинаковой нормализацией. Для части экспериментов снимки при обучении немного поворачиваются и сдвигаются — это **аугментация**. На валидации и тесте таких случайных изменений нет.

**Базовая CNN:** четыре раза применяются свёртка → ReLU → MaxPooling; число каналов растёт от 32 до 256. После этого Global Average Pooling и линейный слой определяют один из четырёх классов.

Для проверки улучшений отдельно используем BatchNorm, Dropout и аугментацию. Дополнительно проверяем **тонкую настройку базовой CNN**: продолжаем её обучение с меньшей скоростью и аугментацией.

### Шаг 7. Создание моделей
Определяем свою CNN и её вариант с BatchNorm/Dropout, затем подготавливаем ResNet18 и MobileNetV3. Последний слой двух готовых сетей заменяется на выход из **четырёх классов**.

In [ ]:
#@title 8. Собственные CNN, ResNet18 и MobileNetV3
NUM_CLASSES = len(CLASS_NAMES)

# Исходная сеть: четыре блока свёртки, затем усреднение и классификация.
class BaselineCNN(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        layers = []
        for in_channels, out_channels in zip([3,32,64,128], [32,64,128,256]):
            layers += [nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                       nn.ReLU(inplace=True), nn.MaxPool2d(2)]
        self.features = nn.Sequential(*layers)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Linear(256, n_classes)
        self.initialize()
    def initialize(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, x):
        x = self.global_avg_pool(self.features(x))
        return self.classifier(torch.flatten(x, 1))

# Второй тип CNN: нормализация и, по выбору, регуляризация Dropout.
class ImprovedCNN(nn.Module):
    def __init__(self, n_classes=4, use_dropout=False):
        super().__init__()
        layers = []
        for in_channels, out_channels in zip([3,32,64,128], [32,64,128,256]):
            layers.extend([nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
                           nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.MaxPool2d(2)])
            if use_dropout:
                layers.append(nn.Dropout2d(0.08))
        self.features = nn.Sequential(*layers)
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.30) if use_dropout else nn.Identity(),
                                        nn.Linear(256, n_classes))
    def forward(self, x):
        return self.classifier(self.global_avg_pool(self.features(x)).flatten(1))

CNN_KEYS = {'cnn_baseline', 'cnn_bn', 'cnn_bn_dropout', 'cnn_bn_dropout_aug'}

def make_model(name, load_imagenet=True):
    if name in {'cnn_baseline', 'cnn_finetuned'}:
        return BaselineCNN(NUM_CLASSES)
    if name in {'cnn_bn', 'cnn_bn_dropout', 'cnn_bn_dropout_aug'}:
        return ImprovedCNN(NUM_CLASSES, use_dropout=name != 'cnn_bn')
    # У предобученных сетей замораживаем признаки и заменяем только последний слой.
    if name == 'resnet18':
        w = models.ResNet18_Weights.DEFAULT if load_imagenet else None
        model = models.resnet18(weights=w)
        for p in model.parameters(): p.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        return model
    if name == 'mobilenet_v3_small':
        w = models.MobileNet_V3_Small_Weights.DEFAULT if load_imagenet else None
        model = models.mobilenet_v3_small(weights=w)
        for p in model.parameters(): p.requires_grad = False
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)
        return model
    raise ValueError(f'Неизвестная архитектура: {name}')

def model_parameters(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)

for name in EPOCHS:
    example = make_model(name, load_imagenet=False)
    total, trainable = model_parameters(example)
    print(f'{name:24s}: {total:>12,d} всего, {trainable:>12,d} обучаемых')
    del example

## 3. Обучение и проверка настроек

На каждой эпохе измеряем **loss** (ошибку) и **accuracy** (точность) на обучении и валидации. Лучшие веса выбираем по **минимальной ошибке валидации** — тест при выборе не используется.

**Early Stopping** прекращает опыт, если ошибка валидации перестала улучшаться. После каждой эпохи сохраняются веса и состояние оптимизатора — обучение можно продолжать после разрыва соединения.

Сначала обучаем собственные CNN и последние классификационные слои ResNet18 и MobileNetV3. Затем отдельно разрешаем обучение части последних свёрточных слоёв предобученных сетей с маленькой скоростью обучения: это может повысить качество на МРТ, но результат определяется валидацией.

### Шаг 8. Функции обучения
Здесь задаются одна эпоха обучения, оценка по валидации, изменение скорости обучения и сохранение лучшей модели. Дополнительный эксперимент `cnn_finetuned` начинает с **готовых весов baseline**, а не с нуля.

In [ ]:
#@title 9. Обучение, сохранение весов и Early Stopping
import gc

PROFILES = {
    'cnn_baseline':         {'lr':1e-3, 'weight_decay':0.0,  'aug':False, 'note':'Conv-ReLU-MaxPool ×4 + GAP'},
    'cnn_bn':               {'lr':1e-3, 'weight_decay':0.0,  'aug':False, 'note':'Добавлена BatchNorm'},
    'cnn_bn_dropout':       {'lr':8e-4, 'weight_decay':1e-4, 'aug':False, 'note':'BatchNorm + Dropout2d/Dropout'},
    'cnn_bn_dropout_aug':   {'lr':8e-4, 'weight_decay':1e-4, 'aug':True,  'note':'BatchNorm + Dropout + Augmentation'},
    'resnet18':             {'lr':1e-3, 'weight_decay':1e-4, 'aug':False, 'note':'ImageNet, обучаем только fc'},
    'mobilenet_v3_small':   {'lr':1e-3, 'weight_decay':1e-4, 'aug':False, 'note':'ImageNet, обучаем только classifier[-1]'},
    'cnn_finetuned':        {'lr':2e-4, 'weight_decay':1e-4, 'aug':True,  'note':'Продолжаем CNN-baseline: меньший LR + аугментация'},
}


def path_best(name): return CKPT_DIR / f'{name}_best.pt'
def path_last(name): return CKPT_DIR / f'{name}_last.pt'

def safe_torch_save(obj, destination):
    temporary = destination.with_suffix('.tmp')
    torch.save(obj, temporary)
    os.replace(temporary, destination)

def budget_seconds_remaining():
    return MAX_TOTAL_MINUTES*60 - (time.monotonic() - SESSION_STARTED)

def make_scaler():
    return torch.amp.GradScaler('cuda', enabled=USE_AMP)

# Один проход по данным. В режиме обучения обновляем веса, при проверке — нет.
def run_epoch(model, dataloader, optimizer=None, scaler=None):
    is_training = optimizer is not None
    model.train(is_training)
    if is_training:
        # Замороженные BatchNorm слои transfer-моделей не обновляют running-statistics.
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                if not any(p.requires_grad for p in module.parameters()):
                    module.eval()
    loss_sum = correct = seen = 0
    criterion = nn.CrossEntropyLoss()
    grad_context = torch.enable_grad() if is_training else torch.inference_mode()
    with grad_context:
        for X, y in dataloader:
            X = X.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            if is_training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits = model(X)
                loss = criterion(logits, y)
            if is_training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            loss_sum += loss.item()*len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            seen += len(y)
    return {'loss': loss_sum/seen, 'acc': correct/seen}


# Обучаем модель, проверяем валидацию и сохраняем best/last.
def train_one(name):
    spec = PROFILES[name]
    has_checkpoint = path_last(name).exists()
    # Если есть checkpoint, ImageNet-веса повторно скачивать не нужно.
    model = make_model(name, load_imagenet=(not has_checkpoint)).to(DEVICE)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                  lr=spec['lr'], weight_decay=spec['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    scaler = make_scaler()
    history = []
    start_epoch = 0
    best_val_loss = float('inf')
    bad_epochs = 0
    early_stopped = False
    if name == 'cnn_finetuned' and not has_checkpoint:
        # Берём уже обученный baseline, вместо обучения новой сети с нуля.
        baseline_path = path_best('cnn_baseline')
        if not baseline_path.exists():
            raise RuntimeError('Сначала обучите базовую CNN, затем запускайте дополнительную настройку.')
        baseline_chk = torch.load(baseline_path, map_location='cpu', weights_only=True)
        if baseline_chk['split_id'] != SPLIT_ID or baseline_chk['image_size'] != IMAGE_SIZE:
            raise RuntimeError('Baseline был обучен на другом разбиении или размере изображений.')
        model.load_state_dict(baseline_chk['model_state'])
        best_val_loss = float(baseline_chk['val_loss'])
        # Нулевая эпоха показывает качество исходных весов до дообучения.
        base_history = pd.read_csv(TABLE_DIR/'cnn_baseline_history.csv')
        base_row = base_history.loc[base_history.val_loss.idxmin()]
        history.append({'epoch': 0, 'train_loss': float(base_row.train_loss),
                        'train_acc': float(base_row.train_acc), 'val_loss': best_val_loss,
                        'val_acc': float(base_row.val_acc), 'lr': spec['lr'], 'epoch_sec': 0.0})
        # Сохраняем исходный baseline как запасной лучший результат.
        safe_torch_save({'model_state': model.state_dict(), 'split_id': SPLIT_ID,
                         'image_size': IMAGE_SIZE, 'epoch': 0,
                         'val_loss': best_val_loss}, path_best(name))
        print('↳ cnn_finetuned: взяты лучшие веса базовой CNN')
    if has_checkpoint:
        previous = torch.load(path_last(name), map_location=DEVICE, weights_only=True)
        if previous['split_id'] != SPLIT_ID or previous['image_size'] != IMAGE_SIZE:
            raise RuntimeError(f'Checkpoint {name} от другого split или IMAGE_SIZE. '
                               'Очистите только соответствующие файлы checkpoint, если намеренно начинаете заново.')
        model.load_state_dict(previous['model_state'])
        optimizer.load_state_dict(previous['optimizer_state'])
        scheduler.load_state_dict(previous['scheduler_state'])
        scaler.load_state_dict(previous.get('scaler_state', {}))
        history = previous['history']
        start_epoch = previous['epoch']
        best_val_loss = previous['best_val_loss']
        bad_epochs = previous['bad_epochs']
        early_stopped = previous.get('early_stopped', False)
        print(f'↳ {name}: продолжаем с эпохи {start_epoch+1}, лучшая val_loss={best_val_loss:.4f}')
    else:
        print(f'↳ {name}: новое обучение, максимум {EPOCHS[name]} эпох')
    if start_epoch >= EPOCHS[name] or early_stopped:
        print(f'✓ {name}: уже завершено, обучение пропущено')
        del model, optimizer, scheduler, scaler
        gc.collect(); torch.cuda.empty_cache()
        return history
    if budget_seconds_remaining() <= 0:
        print('Достигнут предел вычислений; сохранённые эпохи можно продолжить.')
        del model, optimizer, scheduler, scaler
        gc.collect(); torch.cuda.empty_cache()
        return history
    train_loader = loader(train_frame, augmentation=spec['aug'], shuffle=True)
    for ep in range(start_epoch, EPOCHS[name]):
        if budget_seconds_remaining() <= 0:
            print('Достигнут предел вычислений; обучение можно продолжить позднее.')
            break
        started = time.monotonic()
        metrics_train = run_epoch(model, train_loader, optimizer, scaler)
        metrics_val = run_epoch(model, VAL_LOADER)
        scheduler.step(metrics_val['loss'])
        improved = metrics_val['loss'] < best_val_loss - 1e-5
        if improved:
            best_val_loss = metrics_val['loss']
            bad_epochs = 0
            safe_torch_save({'model_state':model.state_dict(), 'split_id':SPLIT_ID,
                             'image_size':IMAGE_SIZE, 'epoch':ep+1, 'val_loss':best_val_loss},
                            path_best(name))
        else:
            bad_epochs += 1
        epoch_s = time.monotonic()-started
        history.append({'epoch':ep+1, 'train_loss':metrics_train['loss'],
                        'train_acc':metrics_train['acc'], 'val_loss':metrics_val['loss'],
                        'val_acc':metrics_val['acc'], 'lr':optimizer.param_groups[0]['lr'],
                        'epoch_sec':epoch_s})
        early_stopped = bad_epochs >= EARLY_STOPPING_PATIENCE
        checkpoint = {'epoch':ep+1, 'model_state':model.state_dict(),
                      'optimizer_state':optimizer.state_dict(), 'scheduler_state':scheduler.state_dict(),
                      'scaler_state':scaler.state_dict(), 'history':history,
                      'best_val_loss':best_val_loss, 'bad_epochs':bad_epochs,
                      'early_stopped':early_stopped, 'split_id':SPLIT_ID, 'image_size':IMAGE_SIZE}
        safe_torch_save(checkpoint, path_last(name))
        pd.DataFrame(history).to_csv(TABLE_DIR/f'{name}_history.csv', index=False)
        print(f'  {name:20} {ep+1:2}/{EPOCHS[name]} | train acc={metrics_train["acc"]:.3f} '
              f'| val acc={metrics_val["acc"]:.3f} | val loss={metrics_val["loss"]:.4f} '
              f'| лучшая эпоха={"да" if improved else "нет"}')
        if early_stopped:
            print(f'  Early Stopping: нет улучшения val_loss {EARLY_STOPPING_PATIENCE} эпохи подряд')
            break
    del model, optimizer, scheduler, scaler, train_loader
    gc.collect(); torch.cuda.empty_cache()
    return history

### Шаг 9. Первые семь экспериментов
Обучаем исходную CNN, три её модификации, два предобученных классификатора и CNN с дополнительной настройкой. После каждой эпохи видны точность и ошибка. **Готовые опыты из Google Drive пропускаются**, поэтому при повторном запуске время не тратится заново.

In [ ]:
#@title 10. Семь экспериментов с CNN и готовыми моделями
EXPERIMENT_ORDER = [
    'cnn_baseline', 'cnn_bn', 'cnn_bn_dropout', 'cnn_bn_dropout_aug',
    'resnet18', 'mobilenet_v3_small', 'cnn_finetuned'
]
# Готовые эксперименты загружаются из Google Drive; заново запускается только недостающее.
ALL_HISTORIES = {}
for experiment in EXPERIMENT_ORDER:
    ALL_HISTORIES[experiment] = train_one(experiment)
print('\nСостояние обучения:')
for experiment in EXPERIMENT_ORDER:
    h = ALL_HISTORIES[experiment]
    completed_epochs = sum(int(row['epoch'] > 0) for row in h)  # начальное состояние (эпоха 0) не считаем
    print(f'  {experiment:24} {completed_epochs}/{EPOCHS[experiment]} эпох, '
          f'best={path_best(experiment).exists()}')


### Шаг 10. Дополнительная настройка ResNet18 и MobileNetV3

Сначала обе сети обучались с **замороженными** свёрточными слоями. Теперь начинаем с их лучших весов, открываем для обучения **последний блок ResNet18** и **несколько последних блоков MobileNetV3**. Используем небольшую скорость обучения и лёгкую аугментацию, чтобы не испортить уже изученные признаки.

Сравниваем качество по валидации: если новая настройка не помогает, сохраняем прежние лучшие веса. Для каждого опыта есть отдельный checkpoint; исходные модели остаются без изменений. Обучение ограничено временем и автоматически пропускается при повторном запуске.

In [ ]:
#@title 11. Точная настройка последних свёрточных блоков
# Дополнительные эксперименты НЕ меняют исходные ResNet18/MobileNetV3.
FINE_TUNE_PARENT = {'resnet18_tuned': 'resnet18',
                    'mobilenet_v3_small_tuned': 'mobilenet_v3_small'}
FINE_TUNE_LR = {'resnet18_tuned': (3e-5, 2e-4),
                'mobilenet_v3_small_tuned': (4e-5, 2e-4)}


def prepare_transfer_model(name):
    parent_name = FINE_TUNE_PARENT[name]
    # На вход подаются именно лучшие веса уже обученного классификатора.
    source = torch.load(path_best(parent_name), map_location='cpu', weights_only=True)
    if source['split_id'] != SPLIT_ID or source['image_size'] != IMAGE_SIZE:
        raise RuntimeError('У исходной модели другое разбиение или размер снимков.')
    net = make_model(parent_name, load_imagenet=False)
    net.load_state_dict(source['model_state'])
    for p in net.parameters():
        p.requires_grad = False
    if name == 'resnet18_tuned':
        backbone_layers, head_layers = list(net.layer4.parameters()), list(net.fc.parameters())
    else:
        # В MobileNetV3 открываем лишь три последних блока извлечения признаков.
        backbone_layers = [p for block in list(net.features.children())[-3:] for p in block.parameters()]
        head_layers = list(net.classifier.parameters())
    for p in backbone_layers + head_layers:
        p.requires_grad = True
    # Для небольшого MRI-датасета статистику BatchNorm оставляем исходной.
    for mod in net.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            for p in mod.parameters():
                p.requires_grad = False
            mod.eval()
    backbone = [p for p in backbone_layers if p.requires_grad]
    head = [p for p in head_layers if p.requires_grad]
    return net.to(DEVICE), backbone, head, source


def train_transfer_finetune(name):
    parent = FINE_TUNE_PARENT[name]
    if not path_best(parent).exists():
        print(f'{name}: нет сохранённой модели {parent}; опыт пропущен.')
        return []
    model, backbone, head, source = prepare_transfer_model(name)
    optimizer = torch.optim.AdamW([
        {'params':backbone, 'lr':FINE_TUNE_LR[name][0]},
        {'params':head, 'lr':FINE_TUNE_LR[name][1]},
    ], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
    scaler = make_scaler()
    last = path_last(name)
    history = []
    best_val_loss = float(source['val_loss'])
    start_epoch = 0
    bad_epochs = 0
    early_stopped = False
    if last.exists():
        saved = torch.load(last, map_location='cpu', weights_only=True)
        if saved['split_id'] != SPLIT_ID or saved['image_size'] != IMAGE_SIZE:
            raise RuntimeError(f'{name}: checkpoint создан на другом разбиении или размере входа.')
        model.load_state_dict(saved['model_state'])
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        scaler.load_state_dict(saved.get('scaler_state', {}))
        start_epoch = int(saved['epoch'])
        best_val_loss = float(saved['best_val_loss'])
        bad_epochs = int(saved['bad_epochs'])
        early_stopped = bool(saved.get('early_stopped', False))
        history = saved['history']
        print(f'↳ {name}: возобновление с {start_epoch+1}-й эпохи')
    else:
        # Нулевая эпоха — результат исходной модели, а не дополнительная эпоха обучения.
        previous = pd.read_csv(TABLE_DIR/f'{parent}_history.csv')
        row = previous.loc[previous.epoch == int(source['epoch'])].iloc[0]
        history = [{'epoch':0, 'train_loss':float(row.train_loss),
                    'train_acc':float(row.train_acc), 'val_loss':best_val_loss,
                    'val_acc':float(row.val_acc), 'lr':FINE_TUNE_LR[name][1], 'epoch_sec':0.0}]
        safe_torch_save({'model_state':model.state_dict(), 'epoch':0,
                         'split_id':SPLIT_ID, 'image_size':IMAGE_SIZE,
                         'val_loss':best_val_loss}, path_best(name))
        pd.DataFrame(history).to_csv(TABLE_DIR/f'{name}_history.csv', index=False)
        print(f'↳ {name}: старт с лучшей модели {parent}; до {TRANSFER_EPOCHS} эпох')
    if start_epoch >= TRANSFER_EPOCHS or early_stopped:
        print(f'✓ {name}: уже завершено')
        del model, optimizer, scheduler, scaler
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        return history
    if budget_seconds_remaining() < 90:
        print(f'{name}: мало времени на обучение; исходные веса уже сохранены.')
        del model, optimizer, scheduler, scaler
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        return history
    train_loader = loader(train_frame, augmentation=True, shuffle=True)
    for ep in range(start_epoch, TRANSFER_EPOCHS):
        # Не начинаем новую эпоху, если времени осталось меньше разумного запаса.
        last_epoch_seconds = history[-1].get('epoch_sec', 0)
        if budget_seconds_remaining() < max(90, 1.35 * last_epoch_seconds + 45):
            print(f'{name}: остановка по пределу времени, состояние сохранено.')
            break
        started = time.monotonic()
        tr = run_epoch(model, train_loader, optimizer, scaler)
        va = run_epoch(model, VAL_LOADER)
        scheduler.step(va['loss'])
        improved = va['loss'] < best_val_loss - 1e-5
        if improved:
            best_val_loss = float(va['loss'])
            bad_epochs = 0
            safe_torch_save({'model_state':model.state_dict(), 'epoch':ep+1,
                             'split_id':SPLIT_ID, 'image_size':IMAGE_SIZE,
                             'val_loss':best_val_loss}, path_best(name))
        else:
            bad_epochs += 1
        history.append({'epoch':ep+1, 'train_loss':tr['loss'], 'train_acc':tr['acc'],
                        'val_loss':va['loss'], 'val_acc':va['acc'],
                        'lr':optimizer.param_groups[0]['lr'],
                        'epoch_sec':time.monotonic()-started})
        early_stopped = bad_epochs >= TRANSFER_PATIENCE
        safe_torch_save({'epoch':ep+1, 'model_state':model.state_dict(),
                         'optimizer_state':optimizer.state_dict(),
                         'scheduler_state':scheduler.state_dict(),
                         'scaler_state':scaler.state_dict(), 'history':history,
                         'best_val_loss':best_val_loss, 'bad_epochs':bad_epochs,
                         'early_stopped':early_stopped, 'split_id':SPLIT_ID,
                         'image_size':IMAGE_SIZE}, last)
        pd.DataFrame(history).to_csv(TABLE_DIR/f'{name}_history.csv', index=False)
        print(f'  {name:25} {ep+1:2}/{TRANSFER_EPOCHS} | '
              f'train acc={tr["acc"]:.3f} | val acc={va["acc"]:.3f} '
              f'| val loss={va["loss"]:.4f} | улучшение={"да" if improved else "нет"}')
        if early_stopped:
            print(f'  {name}: Early Stopping — {TRANSFER_PATIENCE} эпохи без улучшения')
            break
    del model, optimizer, scheduler, scaler, train_loader
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return history

TRANSFER_HISTORIES = {}
for transfer_name in FINE_TUNE_PARENT:
    TRANSFER_HISTORIES[transfer_name] = train_transfer_finetune(transfer_name)
print('Дополнительное обучение завершено или сохранено для продолжения.')


## 4. Графики обучения и сравнение экспериментов

**Accuracy** показывает долю верных ответов, а **loss** — величину ошибки. По кривым валидации видно, на каких эпохах обучение помогало, а где модель переставала улучшаться. Для дообученных сетей **эпоха 0 — это стартовые готовые веса**, её не нужно считать новой эпохой обучения.

### Шаг 11. Графики и таблица настроек
Строим кривые валидации и таблицу для всех вариантов CNN, ResNet18 и MobileNetV3. Сравниваем, помогли ли BatchNorm, Dropout, аугментация и дообучение. Лучшую эпоху выбираем по **минимальному val_loss**.

In [ ]:
#@title 12. Графики валидации и сравнение экспериментов
ALL_EXPERIMENT_ORDER = EXPERIMENT_ORDER + list(FINE_TUNE_PARENT)
history_rows = []
for name in ALL_EXPERIMENT_ORDER:
    p = TABLE_DIR/f'{name}_history.csv'
    if not p.exists():
        continue
    frame = pd.read_csv(p)
    if frame.empty: continue
    frame['model'] = name
    history_rows.append(frame)
if not history_rows:
    raise RuntimeError('Нет сохранённых историй обучения. Начните с ячейки 10.')
history_all = pd.concat(history_rows, ignore_index=True)
# По графикам можно видеть стабильность обучения и колебания val_loss.
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for name in ALL_EXPERIMENT_ORDER:
    part = history_all[history_all.model == name]
    if part.empty: continue
    axes[0].plot(part.epoch, part.val_acc, marker='o', ms=3, label=name)
    axes[1].plot(part.epoch, part.val_loss, marker='o', ms=3, label=name)
axes[0].set(title='Accuracy на валидации', xlabel='Эпоха', ylabel='Accuracy')
axes[1].set(title='Cross-entropy на валидации', xlabel='Эпоха', ylabel='Loss')
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR/'02_training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

ablation_rows = []
for name in ALL_EXPERIMENT_ORDER:
    part = history_all[history_all.model==name]
    if part.empty: continue
    best = part.loc[part.val_loss.idxmin()]
    ablation_rows.append({'Модель':name, 'Изменение':(PROFILES[name]['note'] if name in PROFILES else 'Тонкая настройка последних свёрточных блоков'),
                          'Эпох':int((part.epoch > 0).sum()), 'Best val acc (%)':round(best.val_acc*100,2),
                          'Best val loss':round(best.val_loss,4),
                          'Лучшая эпоха':('стартовая' if int(best.epoch) == 0 else int(best.epoch))})
ablation_table = pd.DataFrame(ablation_rows)
ablation_table.to_csv(TABLE_DIR/'validation_experiments.csv', index=False)
display(ablation_table)
# Сравниваем исходные и дополнительно настроенные версии по ВАЛИДАЦИИ.
for tuned, parent in FINE_TUNE_PARENT.items():
    pair = ablation_table[ablation_table['Модель'].isin([parent, tuned])]
    if len(pair) == 2:
        print(f'Валидация {parent}: {pair.iloc[0]["Best val loss"]:.4f} → '
              f'{pair.iloc[1]["Best val loss"]:.4f} (после настройки)')

## 5. Итоговая оценка на тесте

Загружаем лучшие сохранённые веса и проверяем **четыре итоговые модели**: исходную CNN, дообученную CNN, ResNet18 и MobileNetV3. Для двух предобученных сетей берём лучшую версию после дополнительной настройки (если этап успел выполниться).

Проверяем все модели на **одних и тех же изображениях**. Точные повторы в тестовой папке удаляем **только при оценке**, не изменяя старое обучение.

- **Accuracy** — доля правильных ответов.
- **Precision** — какая часть ответов данного класса верна.
- **Recall** — какая часть настоящих снимков данного класса распознана.
- **Macro-F1** — среднее качество по классам с одинаковым весом.

Тест служит **только для окончательной проверки**, а не выбора настроек.

### Шаг 12. Итоговые метрики
Для каждой модели считаем Accuracy, Precision, Recall и Macro-F1. У всех одинаковый тест, из которого исключены только **точные дубликаты**. Счётчики по классам и все предсказания сохраняем в таблицы.

In [ ]:
#@title 13. Тестирование четырёх моделей и итоговые метрики
# Если новый эксперимент прерван до старта, берём сохранённые исходные версии.
FINAL_NAMES = [
    'cnn_baseline', 'cnn_finetuned',
    'resnet18_tuned' if path_best('resnet18_tuned').exists() else 'resnet18',
    'mobilenet_v3_small_tuned' if path_best('mobilenet_v3_small_tuned').exists() else 'mobilenet_v3_small'
]

# Загружаем только лучший checkpoint; тестовые данные не участвуют в обучении.
def predict_from_best(name):
    chk_path = path_best(name)
    if not chk_path.exists():
        print(f'⚠ {name}: пока нет best checkpoint, модель не будет участвовать в сравнении.')
        return None
    checkpoint = torch.load(chk_path, map_location='cpu', weights_only=True)
    if checkpoint['split_id'] != SPLIT_ID or checkpoint['image_size'] != IMAGE_SIZE:
        raise RuntimeError(f'Некорректный checkpoint {name}: изменилось разбиение или размер изображений.')
    model = make_model(FINE_TUNE_PARENT.get(name, name), load_imagenet=False).to(DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    truths, preds, confs = [], [], []
    with torch.inference_mode():
        for X, y in TEST_LOADER:
            X = X.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits = model(X)
            probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
            truths.extend(y.tolist())
            preds.extend(probs.argmax(axis=1).tolist())
            confs.extend(probs.max(axis=1).tolist())
    del model
    gc.collect(); torch.cuda.empty_cache()
    return np.array(truths), np.array(preds), np.array(confs)

TEST_PREDICTIONS = {}
summary_rows = []
wide_preds = pd.DataFrame({'image_path':test_frame.rel_path,
                           'true_class':test_frame.label})
for name in FINAL_NAMES:
    result = predict_from_best(name)
    if result is None: continue
    yt, yp, confidence = result
    assert np.array_equal(yt, test_frame.label.map(CLASS_TO_IDX).to_numpy())
    TEST_PREDICTIONS[name] = (yt, yp, confidence)
    wide_preds[f'{name}_pred'] = [CLASS_NAMES[int(k)] for k in yp]
    wide_preds[f'{name}_confidence'] = np.round(confidence, 4)
    acc = accuracy_score(yt, yp)
    summary_rows.append({'Модель':name, 'Accuracy (%)':round(acc*100,2),
                         'Macro F1 (%)':round(f1_score(yt,yp,average='macro',zero_division=0)*100,2),
                         'Macro Precision (%)':round(precision_score(yt,yp,average='macro',zero_division=0)*100,2),
                         'Macro Recall (%)':round(recall_score(yt,yp,average='macro',zero_division=0)*100,2),
                         'Test images':len(yt)})
    report = classification_report(yt, yp, labels=list(range(NUM_CLASSES)),
                                   target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(TABLE_DIR/f'{name}_classification_report.csv')
    print('\n'+'='*75+'\n'+name+'\n'+'='*75)
    print(classification_report(yt, yp, labels=list(range(NUM_CLASSES)),
                                target_names=CLASS_NAMES, zero_division=0))

if not summary_rows:
    raise RuntimeError('Не найдено ни одного checkpoint. Выполните обучение в ячейке 10.')
test_summary = pd.DataFrame(summary_rows).sort_values('Macro F1 (%)', ascending=False)
test_summary.to_csv(TABLE_DIR/'final_test_comparison.csv', index=False)
wide_preds.to_csv(TABLE_DIR/'all_test_predictions.csv', index=False)
display(test_summary)
if len(summary_rows)<4:
    print('⚠ Итоговое сравнение НЕ ПОЛНОЕ: не все четыре модели обучены. '
          'Сохраните результаты и продолжите обучение позднее.')

### Шаг 13. Матрицы ошибок
На диагонали — правильные ответы; вне диагонали — ошибки. Так видно, например, насколько часто сеть путает глиому и менингиому.

In [ ]:
#@title 14. Матрицы ошибок классификации
# Диагональ матрицы показывает верные ответы, остальные клетки — ошибки.
fig, axs = plt.subplots(2, 2, figsize=(14, 12))
for ax, name in zip(axs.flat, FINAL_NAMES):
    if name not in TEST_PREDICTIONS:
        ax.axis('off'); ax.set_title(name+' (пока не обучена)'); continue
    yt, yp, _ = TEST_PREDICTIONS[name]
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=[CLASS_RU[c] for c in CLASS_NAMES],
                yticklabels=[CLASS_RU[c] for c in CLASS_NAMES])
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Предсказанный класс')
    ax.set_ylabel('Истинный класс')
    ax.tick_params(axis='x', rotation=25)
    ax.tick_params(axis='y', rotation=0)
plt.suptitle('Матрицы ошибок на независимой тестовой выборке', fontsize=15)
plt.tight_layout()
plt.savefig(FIG_DIR/'03_confusion_matrices.png', dpi=170, bbox_inches='tight')
plt.show()

### Шаг 14. Примеры предсказаний
На **восьми увеличенных снимках** показываем истинный класс и решения четырёх моделей. Зелёная отметка означает правильный ответ, красная — ошибку. Часть примеров верная для всех сетей, часть содержит ошибки.

In [ ]:
#@title 15. Крупные примеры правильных и ошибочных предсказаний
if len(TEST_PREDICTIONS) != 4:
    print('Не все четыре модели доступны, показываем полученные предсказания.')
y_true = test_frame.label.map(CLASS_TO_IDX).to_numpy()
available = list(TEST_PREDICTIONS)
wrong_any = np.array([any(TEST_PREDICTIONS[m][1][i] != y_true[i] for m in available)
                      for i in range(len(test_frame))])
correct_all = np.flatnonzero(~wrong_any)
# Сначала несколько полностью верных примеров, затем — ошибки разных классов.
rng = np.random.default_rng(SEED)
selected = rng.choice(correct_all, size=min(2, len(correct_all)), replace=False).tolist() if len(correct_all) else []
for cls in range(NUM_CLASSES):
    pool = np.flatnonzero(wrong_any & (y_true == cls))
    if len(pool):
        chosen = rng.choice(pool, size=min(2, len(pool)), replace=False).tolist()
        selected.extend([int(i) for i in chosen if i not in selected])
    if len(selected) >= 8: break
selected = selected[:8]
if len(selected) < 8:
    remaining = np.setdiff1d(np.arange(len(test_frame)), np.array(selected, dtype=int))
    selected.extend(rng.choice(remaining, size=min(8-len(selected),len(remaining)), replace=False).tolist())

# Каждому снимку выделена своя широкая текстовая область: подписи остаются читаемыми.
from matplotlib.gridspec import GridSpec
rows = max(1, (len(selected)+1)//2)
fig = plt.figure(figsize=(17, 3.3*rows))
gs = GridSpec(rows, 4, figure=fig, width_ratios=[1,1.35,1,1.35], wspace=0.04, hspace=0.35)
short_names = {'cnn_baseline':'CNN базовая', 'cnn_finetuned':'CNN улучш.',
               'resnet18':'ResNet18', 'resnet18_tuned':'ResNet18+настройка',
               'mobilenet_v3_small':'MobileNetV3',
               'mobilenet_v3_small_tuned':'MobileNetV3+настройка'}
for pos, idx in enumerate(selected):
    r, c = divmod(pos, 2)
    image_ax = fig.add_subplot(gs[r,2*c]); text_ax = fig.add_subplot(gs[r,2*c+1])
    item = test_frame.iloc[idx]
    with Image.open(DATA_ROOT/item.rel_path) as im:
        image_ax.imshow(im.convert('RGB'))
    image_ax.axis('off'); text_ax.axis('off')
    text_ax.text(0.01,0.92,f'Истина: {CLASS_RU[item.label]}',weight='bold',fontsize=11,
                 transform=text_ax.transAxes,va='top')
    for j, m in enumerate(available):
        predicted = CLASS_NAMES[int(TEST_PREDICTIONS[m][1][idx])]
        ok = predicted == item.label
        text_ax.text(0.01,0.76-0.16*j,
                     f'{"✓" if ok else "✗"}  {short_names[m]}:\n    {CLASS_RU[predicted]}',
                     color='#176b38' if ok else '#ad3030',fontsize=10.3,
                     transform=text_ax.transAxes,va='top')
plt.suptitle('Примеры классификации МРТ: верные ответы и ошибки',fontsize=15, y=1.005)
plt.savefig(FIG_DIR/'04_correct_and_wrong_examples.png',dpi=165,bbox_inches='tight')
plt.show()
print(f'Хотя бы одна модель ошиблась: {int(wrong_any.sum())} из {len(test_frame)} уникальных тестовых снимков.')


## 6. Выводы и сохранение результатов

После сравнения строим короткий анализ по классам: **какие снимки распознаются лучше**, **какие два класса чаще путаются**, **насколько изменилась CNN** по сравнению с базовой. Это объясняет результаты точнее одной общей цифры Accuracy.

Графики, таблицы, все предсказания и описание ограничений сохраняются в Google Drive. Снимки могут относиться к одному пациенту; без идентификаторов пациентов гарантировать полную независимость данных нельзя.

### Шаг 15. Анализ ошибок и материалы
Находим класс с самой низкой полнотой распознавания (`Recall`) и самую частую путаницу для лучшей модели. Сохраняем краткое описание опыта; затем упаковываем таблицы и рисунки в ZIP. Тяжёлые веса нейросетей сохраняются отдельно.

In [ ]:
#@title 16. Краткий анализ результатов
train_counts = {key:len(manifest[manifest.split==key]) for key in ('train','val')}
train_counts['test_unique'] = len(test_frame)
training_details = ', '.join(f'{k}={v}' for k,v in train_counts.items())
measurements = test_summary.to_markdown(index=False)
ablation_md = ablation_table.to_markdown(index=False)
completed = len(TEST_PREDICTIONS) == len(FINAL_NAMES)

# Считаем ошибки лучшей модели: не только общую точность, но и проблемы отдельных классов.
leader = test_summary.iloc[0]['Модель']
true_best, pred_best, _ = TEST_PREDICTIONS[leader]
leader_cm = confusion_matrix(true_best, pred_best, labels=list(range(NUM_CLASSES)))
per_class_recall = np.diag(leader_cm) / np.maximum(leader_cm.sum(axis=1), 1)
weakest = int(np.argmin(per_class_recall))
confusions = leader_cm.copy()
np.fill_diagonal(confusions, 0)
wrong_from, wrong_to = np.unravel_index(np.argmax(confusions), confusions.shape)
base_accuracy = float(test_summary.loc[test_summary['Модель']=='cnn_baseline','Accuracy (%)'].iloc[0])
cnn_accuracy = float(test_summary.loc[test_summary['Модель']=='cnn_finetuned','Accuracy (%)'].iloc[0])
delta_cnn = cnn_accuracy - base_accuracy
analysis_lines = [
    f'Лучшая по Macro-F1 модель: {leader} (Accuracy={test_summary.iloc[0]["Accuracy (%)"]:.2f}%).',
    f'Разница между дообученной и базовой CNN: {delta_cnn:+.2f} процентного пункта Accuracy.',
    f'Самый сложный класс для лидера: {CLASS_RU[CLASS_NAMES[weakest]]}; '
    f'Recall={per_class_recall[weakest]*100:.1f}% '
    f'({leader_cm[weakest,weakest]} из {leader_cm[weakest].sum()}).',
    f'Наиболее частая ошибка лидера: {CLASS_RU[CLASS_NAMES[wrong_from]]} '
    f'→ {CLASS_RU[CLASS_NAMES[wrong_to]]}, случаев: {int(confusions[wrong_from,wrong_to])}.'
]
print('\n'.join(analysis_lines))
(TABLE_DIR/'short_error_analysis.txt').write_text('\n'.join(analysis_lines),encoding='utf-8')

report_body = f"""# Лабораторная работа № 3. Вариант 24

**Тема:** Классификация МРТ головного мозга свёрточными нейронными сетями.

## Цель
Обучить свою CNN, проверить её улучшения, сравнить с ResNet18 и MobileNetV3 и объяснить ошибки классификации.

## Данные и обработка
Источник: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset

Классы: {', '.join(CLASS_NAMES)}.
Разделение: {training_details}.
Удалено точных дубликатов из исходного теста: {removed_test_duplicates} (из {len(test_frame_original)}).
Seed: {SEED}; снимки: {IMAGE_SIZE}×{IMAGE_SIZE}×3; пакет: {BATCH_SIZE}.
Валидацию выделили только из Training, а тест не использовали при настройке моделей.
По SHA-256 удалили точные повторы из обучения; ID разбиения: {SPLIT_ID}.

## Базовая CNN и улучшения
CNN состоит из четырёх блоков Conv2d(3×3)–ReLU–MaxPool, каналов 32/64/128/256,
GlobalAveragePooling и классификатора на четыре класса.
Сравнили BatchNorm, Dropout и аугментацию. Затем продолжили обучение базовой CNN
с меньшим learning rate. Использовали AdamW, CrossEntropyLoss, ReduceLROnPlateau и Early Stopping.

## Сравнение по валидации
{ablation_md}

## Предобученные модели
ResNet18 и MobileNetV3-Small получили веса ImageNet. Сначала обучали только классификаторы,
затем отдельно дообучали последние свёрточные блоки с малым learning rate.
Для финальной оценки каждой сети используем лучший checkpoint по val_loss, включая исходное состояние.

## Итоговые результаты на уникальных снимках теста
{measurements}

Полнота сравнения: {'все четыре модели протестированы' if completed else 'часть моделей пока не завершена'}.

## Конкретный анализ ошибок
{chr(10).join('- '+line for line in analysis_lines)}

Графики: figures/02_training_curves.png и figures/03_confusion_matrices.png.
Примеры: figures/04_correct_and_wrong_examples.png.
Ответы каждой модели: tables/all_test_predictions.csv.

## Ограничения
Точный SHA-256 не обнаруживает почти одинаковые снимки или фотографии одного пациента.
Идентификаторов пациентов нет, поэтому полную независимость train/val/test подтвердить нельзя.
Классификация учебная и не является медицинским заключением.

## Вывод
Эксперименты проведены на фиксированной валидации и едином тесте без точных дублей.
Лучшие веса каждой модели выбраны по валидации; тест использован для итогового сравнения архитектур.
"""
(OUT_DIR/'REPORT_MATERIALS.md').write_text(report_body, encoding='utf-8')
print('Создано описание эксперимента и анализ ошибок:', OUT_DIR/'REPORT_MATERIALS.md')


In [ ]:
#@title 17. Сохранение таблиц, графиков и текста в архив
archive = OUT_DIR/'RESULTS_FOR_REPORT.zip'
# В архив включаем только материалы анализа, а тяжёлые веса оставляем в checkpoints/.
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    z.write(OUT_DIR/'REPORT_MATERIALS.md', arcname='REPORT_MATERIALS.md')
    for directory in (FIG_DIR, TABLE_DIR):
        for path in sorted(directory.rglob('*')):
            if path.is_file():
                z.write(path, arcname=str(path.relative_to(OUT_DIR)))
print('Готово!')
print('Архив с графиками, таблицами и текстом отчёта:', archive)
print('Отдельно веса моделей: ', CKPT_DIR)
